<a href="https://colab.research.google.com/github/ManhattanCB01-nyc/CrashVehiclesDataPipeline/blob/main/intersection_collision_counts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manhattan CB1 Collisions — Same-Intersection Counts

Counts collisions per intersection from `manhattan_cb1_collisions_2018plus.xlsx`
(the **Crashes** sheet), treating two GPS coordinates as *the same intersection*
if they round to the same value at 4 decimal places (~11 m — small enough to
merge GPS noise between repeat reports of one intersection, large enough not to
merge two different nearby intersections).

Coordinates for the same real-world intersection are almost never bit-identical
— e.g. Broadway & Canal St shows up as both `(40.719395, -74.001890)` and
`(40.719396, -74.001883)` in this file — so grouping on the raw floats would
undercount. Rounding fixes that.

**Before running:** upload `manhattan_cb1_collisions_2018plus.xlsx` to your
Google Drive and set `DRIVE_FILE_PATH` in the second code cell. Fill in
`PROJECT_ID` in the BigQuery cell near the end with your own GCP project ID.

Steps: mount Drive → load & clean the Crashes sheet → round & group
coordinates → map the results → save a CSV back to Drive → load the summary
table into BigQuery.


In [40]:
# Install packages not preinstalled in Colab
!pip install -q openpyxl folium pandas-gbq


In [41]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
# EDIT  path to the file inside your mounted Drive
DRIVE_FILE_PATH = "/content/drive/MyDrive/manhattan_cb1_collisions/data/manhattan_cb1_collisions_2018plus_copy.xlsx"

# How close two coordinates must be to count as "the same" intersection.
# 4 decimal places of lat/lon = 90 m circle.
ROUND_DECIMALS = 4


In [43]:
import pandas as pd
import numpy as np

# The header row isn't always in the same place in every export of this file
# (in at least one copy it ends up as the LAST row instead of the first), so
# read without assuming a header and detect it by content instead of position.
raw = pd.read_excel(DRIVE_FILE_PATH, sheet_name="Crashes", header=None)

EXPECTED_HEADER_TOKENS = {"crash_date", "borough", "latitude", "longitude", "collision_id"}

header_row_idx = None
for i, row in raw.iterrows():
    values = {str(v).strip().lower() for v in row if pd.notna(v)}
    if EXPECTED_HEADER_TOKENS.issubset(values):
        header_row_idx = i
        break

if header_row_idx is None:
    # No embedded header row found — assume row 0 is already a normal header.
    df = pd.read_excel(DRIVE_FILE_PATH, sheet_name="Crashes")
else:
    header = [str(v).strip().lower() if pd.notna(v) else f"col_{j}" for j, v in raw.loc[header_row_idx].items()]
    df = raw.drop(index=header_row_idx).reset_index(drop=True)
    df.columns = header

print(f"Loaded {len(df):,} crash rows. Header row found at index: {header_row_idx}")
df.head()


Loaded 4,296 crash rows. Header row found at index: 0


,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,off_street_name,number_of_persons_injured,...,collision_id,vehicle_type_code1,vehicle_type_code2,cross_street_name,contributing_factor_vehicle_3,vehicle_type_code_3,contributing_factor_vehicle_4,vehicle_type_code_4,contributing_factor_vehicle_5,vehicle_type_code_5
0,2020-01-01 00:00:00,4:56,MANHATTAN,10007,40.7163,-74.01097,"{'latitude': '40.7163', 'longitude': '-74.01097'}",GREENWICH STREET,CHAMBERS STREET,1,...,4267803,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02 00:00:00,14:00,MANHATTAN,10005,40.705307,-74.00903,"{'latitude': '40.705307', 'longitude': '-74.00...",BEAVER STREET,HANOVER STREET,0,...,4268366,Station Wagon/Sport Utility Vehicle,Dump,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-02 00:00:00,21:30,MANHATTAN,10007,40.711933,-74.00985,"{'latitude': '40.711933', 'longitude': '-74.00...",NaN,NaN,1,...,4271936,Sedan,Bike,30 VESEY STREET,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-03 00:00:00,16:44,MANHATTAN,10004,40.704952,-74.01566,"{'latitude': '40.704952', 'longitude': '-74.01...",BATTERY PLACE,WASHINGTON STREET,0,...,4268675,Station Wagon/Sport Utility Vehicle,Taxi,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-03 00:00:00,7:10,MANHATTAN,10013,40.72135,-74.00465,"{'latitude': '40.72135', 'longitude': '-74.004...",CANAL STREET,WEST BROADWAY,0,...,4268678,Sedan,Sedan,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# Keep only rows with usable coordinates
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

geo = df.dropna(subset=["latitude", "longitude"]).copy()
geo = geo[(geo["latitude"] != 0) & (geo["longitude"] != 0)]

print(f"{len(geo):,} of {len(df):,} rows have valid coordinates.")


4,296 of 4,296 rows have valid coordinates.


In [45]:
# Round coordinates so GPS noise around one real intersection collapses
# into a single group, then count collisions per group.
geo["lat_rounded"] = geo["latitude"].round(ROUND_DECIMALS)
geo["lon_rounded"] = geo["longitude"].round(ROUND_DECIMALS)

def most_common(series):
    s = series.dropna()
    return s.mode().iloc[0] if not s.mode().empty else None

agg_kwargs = {"collision_count": ("collision_id", "count")}
if "on_street_name" in geo.columns:
    agg_kwargs["on_street_name"] = ("on_street_name", most_common)
if "off_street_name" in geo.columns:
    agg_kwargs["off_street_name"] = ("off_street_name", most_common)

intersections = (
    geo.groupby(["lat_rounded", "lon_rounded"])
       .agg(**agg_kwargs)
       .reset_index()
       .sort_values("collision_count", ascending=False)
       .reset_index(drop=True)
)

print(f"{len(intersections):,} distinct intersections found.")
intersections.head(20)


1,321 distinct intersections found.


,lat_rounded,lon_rounded,collision_count,on_street_name,off_street_name
0,40.7194,-74.0019,69,CANAL STREET,BROADWAY
1,40.7184,-74.0005,67,CANAL STREET,LAFAYETTE STREET
2,40.7214,-74.0046,60,CANAL STREET,WEST BROADWAY
3,40.7102,-74.0011,54,PEARL STREET,ROBERT F WAGNER PLACE
4,40.7095,-74.0017,53,PEARL STREET,DOVER STREET
5,40.7226,-74.0063,53,CANAL STREET,VARICK STREET
6,40.7180,-74.0000,52,CANAL STREET,CENTRE STREET
7,40.7152,-74.0134,46,WEST STREET,MURRAY STREET
8,40.7172,-74.0129,44,WEST STREET,CHAMBERS STREET
9,40.7131,-74.0041,40,CENTRE STREET,CHAMBERS STREET


**Note on this approach:** rounding is a simple grid — two points on opposite
sides of a rounding boundary (e.g. `40.70995` vs `40.71005`) could in rare
cases land in different groups despite being close. Given this file's actual
GPS spread (median well under 2 m, 95th percentile ~6 m for repeat visits to
the same named intersection), 4-decimal rounding is a solid match. If a later
pass finds boundary splitting is a real problem, swap this step for
distance-based clustering (e.g. `sklearn.cluster.DBSCAN` with a haversine
metric) instead of grid rounding.


In [46]:
import folium

center_lat = intersections["lat_rounded"].mean()
center_lon = intersections["lon_rounded"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="cartodbpositron")

max_count = intersections["collision_count"].max()

for _, row in intersections.iterrows():
    label_bits = [f"Collisions: {row['collision_count']}"]
    if "on_street_name" in intersections.columns and pd.notna(row.get("on_street_name")):
        label_bits.append(f"On: {row['on_street_name']}")
    if "off_street_name" in intersections.columns and pd.notna(row.get("off_street_name")):
        label_bits.append(f"Cross: {row['off_street_name']}")

    folium.CircleMarker(
        location=[row["lat_rounded"], row["lon_rounded"]],
        radius=3 + 12 * (row["collision_count"] / max_count) ** 0.5,
        color="#d62728",
        fill=True,
        fill_opacity=0.6,
        popup="<br>".join(label_bits),
    ).add_to(m)

m


In [47]:
# Save the summary table as a CSV back into Drive
OUTPUT_CSV_PATH = "/content/drive/MyDrive/manhattan_cb1_collisions/manhattan_cb1_intersection_collision_counts.csv"

intersections.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Saved {len(intersections):,} rows to {OUTPUT_CSV_PATH}")


Saved 1,321 rows to /content/drive/MyDrive/manhattan_cb1_collisions/manhattan_cb1_intersection_collision_counts.csv


In [48]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "stable-liberty-426016-d2"

BQ_DATASET = "new_dataset"
BQ_TABLE = "collisioncount"

In [49]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.DatasetReference(PROJECT_ID, BQ_DATASET)
try:
    client.get_dataset(dataset_ref)
except Exception:
    client.create_dataset(bigquery.Dataset(dataset_ref))
    print(f"Created dataset {BQ_DATASET}")

table_id = f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = client.load_table_from_dataframe(intersections, table_id, job_config=job_config)
job.result()

print(f"Loaded {len(intersections):,} rows into {table_id}")

Loaded 1,321 rows into stable-liberty-426016-d2.new_dataset.collisioncount


In [50]:
from google.cloud import bigquery

# Construct a BigQuery client object.
client = bigquery.Client(project=PROJECT_ID)

# Query the table, ordering by collision_count in descending order
query = f"""
    SELECT lat_rounded, lon_rounded, collision_count, on_street_name, off_street_name
    FROM `{table_id}`
    ORDER BY collision_count DESC
    LIMIT 12
"""

query_job = client.query(query)
results = query_job.to_dataframe()

display(results)

,lat_rounded,lon_rounded,collision_count,on_street_name,off_street_name
0,40.7194,-74.0019,69,CANAL STREET,BROADWAY
1,40.7184,-74.0005,67,CANAL STREET,LAFAYETTE STREET
2,40.7214,-74.0046,60,CANAL STREET,WEST BROADWAY
3,40.7102,-74.0011,54,PEARL STREET,ROBERT F WAGNER PLACE
4,40.7226,-74.0063,53,CANAL STREET,VARICK STREET
5,40.7095,-74.0017,53,PEARL STREET,DOVER STREET
6,40.7180,-74.0000,52,CANAL STREET,CENTRE STREET
7,40.7152,-74.0134,46,WEST STREET,MURRAY STREET
8,40.7172,-74.0129,44,WEST STREET,CHAMBERS STREET
9,40.7131,-74.0041,40,CENTRE STREET,CHAMBERS STREET


In [51]:
top_10_intersections = intersections.head(10)

# Define a new path for the top 10 intersections CSV
TOP_10_OUTPUT_CSV_PATH = "/content/drive/MyDrive/manhattan_cb1_collisions/manhattan_cb1_top_10_intersections.csv"

# Save the top 10 intersections to CSV
top_10_intersections.to_csv(TOP_10_OUTPUT_CSV_PATH, index=False)
print(f"Saved {len(top_10_intersections):,} rows to {TOP_10_OUTPUT_CSV_PATH}")

display(top_10_intersections)

Saved 10 rows to /content/drive/MyDrive/manhattan_cb1_collisions/manhattan_cb1_top_10_intersections.csv


,lat_rounded,lon_rounded,collision_count,on_street_name,off_street_name
0,40.7194,-74.0019,69,CANAL STREET,BROADWAY
1,40.7184,-74.0005,67,CANAL STREET,LAFAYETTE STREET
2,40.7214,-74.0046,60,CANAL STREET,WEST BROADWAY
3,40.7102,-74.0011,54,PEARL STREET,ROBERT F WAGNER PLACE
4,40.7095,-74.0017,53,PEARL STREET,DOVER STREET
5,40.7226,-74.0063,53,CANAL STREET,VARICK STREET
6,40.7180,-74.0000,52,CANAL STREET,CENTRE STREET
7,40.7152,-74.0134,46,WEST STREET,MURRAY STREET
8,40.7172,-74.0129,44,WEST STREET,CHAMBERS STREET
9,40.7131,-74.0041,40,CENTRE STREET,CHAMBERS STREET


In [52]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

# Define a new table name for the top 10 intersections
TOP_10_BQ_TABLE = "top_10_collision_intersections"

top_10_table_id = f"{PROJECT_ID}.{BQ_DATASET}.{TOP_10_BQ_TABLE}"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = client.load_table_from_dataframe(top_10_intersections, top_10_table_id, job_config=job_config)
job.result()

print(f"Loaded {len(top_10_intersections):,} rows into {top_10_table_id}")

Loaded 10 rows into stable-liberty-426016-d2.new_dataset.top_10_collision_intersections


You can verify the new BigQuery table by querying it:

In [53]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

# Query the new top 10 table
query = f"""
    SELECT lat_rounded, lon_rounded, collision_count, on_street_name, off_street_name
    FROM `{top_10_table_id}`
    ORDER BY collision_count DESC
"""

query_job = client.query(query)
results_top_10 = query_job.to_dataframe()

display(results_top_10)

,lat_rounded,lon_rounded,collision_count,on_street_name,off_street_name
0,40.7194,-74.0019,69,CANAL STREET,BROADWAY
1,40.7184,-74.0005,67,CANAL STREET,LAFAYETTE STREET
2,40.7214,-74.0046,60,CANAL STREET,WEST BROADWAY
3,40.7102,-74.0011,54,PEARL STREET,ROBERT F WAGNER PLACE
4,40.7095,-74.0017,53,PEARL STREET,DOVER STREET
5,40.7226,-74.0063,53,CANAL STREET,VARICK STREET
6,40.7180,-74.0000,52,CANAL STREET,CENTRE STREET
7,40.7152,-74.0134,46,WEST STREET,MURRAY STREET
8,40.7172,-74.0129,44,WEST STREET,CHAMBERS STREET
9,40.7131,-74.0041,40,CENTRE STREET,CHAMBERS STREET


In [54]:
import pandas as pd
from google.cloud import bigquery

# Ensure the variable 'intersections' exists from previous cells
try:
    # 1. Get the top 10 intersections from the existing results
    top_10_df = intersections.head(10)

    # 2. Save to Google Drive
    TOP_10_DRIVE_PATH = "/content/drive/MyDrive/manhattan_cb1_collisions/top_10_intersections.csv"
    top_10_df.to_csv(TOP_10_DRIVE_PATH, index=False)
    print(f"Successfully saved top 10 rows to Drive: {TOP_10_DRIVE_PATH}")

    # 3. Save to BigQuery
    client = bigquery.Client(project=PROJECT_ID)
    TOP_10_TABLE_ID = f"{PROJECT_ID}.{BQ_DATASET}.top_10_collisions"

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    job = client.load_table_from_dataframe(top_10_df, TOP_10_TABLE_ID, job_config=job_config)
    job.result()  # Wait for the job to complete

    print(f"Successfully loaded top 10 rows into BigQuery: {TOP_10_TABLE_ID}")

    # Display the top 10 data
    display(top_10_df)
except NameError:
    print("Error: The 'intersections' DataFrame was not found. Please run the cells above to load and process the collision data first.")

Successfully saved top 10 rows to Drive: /content/drive/MyDrive/manhattan_cb1_collisions/top_10_intersections.csv
Successfully loaded top 10 rows into BigQuery: stable-liberty-426016-d2.new_dataset.top_10_collisions


,lat_rounded,lon_rounded,collision_count,on_street_name,off_street_name
0,40.7194,-74.0019,69,CANAL STREET,BROADWAY
1,40.7184,-74.0005,67,CANAL STREET,LAFAYETTE STREET
2,40.7214,-74.0046,60,CANAL STREET,WEST BROADWAY
3,40.7102,-74.0011,54,PEARL STREET,ROBERT F WAGNER PLACE
4,40.7095,-74.0017,53,PEARL STREET,DOVER STREET
5,40.7226,-74.0063,53,CANAL STREET,VARICK STREET
6,40.7180,-74.0000,52,CANAL STREET,CENTRE STREET
7,40.7152,-74.0134,46,WEST STREET,MURRAY STREET
8,40.7172,-74.0129,44,WEST STREET,CHAMBERS STREET
9,40.7131,-74.0041,40,CENTRE STREET,CHAMBERS STREET
